In [1]:
#imports

import numpy as np

In [ ]:
# TODOs:
# - compute d once per frame, from the FFT
# - add yin_tracks(x, hop), where hop is the period at which yin_frame is called, which returns (time, f0)
# - evaluate on URMP tracks and own recordings against librosa's pYIN

In [3]:
# simple autocorrelation function (prone to "too high" mis-detections ??)

def acor(x, tau, W, t0=0):
    '''
    inputs:
        x[t]    - time-domain discrete audio signal (assuming 1D-np-array), 
        tau     - lag / candidate period, measured in samples
        t0      - starting sample of window W
        W       - window of autocorrelation function

    returns:
        autocorrelation value r_t0(tau) = Σ_{j=0}^{W-1} x[t+j] · x[t+j+tau]
        r_t0(0) will always yield the global maximum (autocorrelation is maximized when comparing to itself)
    '''

    #r_t0 = 0
    #for j in range(W):
    #    r_0 += x[t0+j]*x[t0+j+tau]

    # need to convert to float64 to resolve any wraparound issue with 16-bit-WAV-audio (int16) files
    # the largest int16 product is ((-2)^15)*((-2)^15) = 2^30, with one bit being the sign bit
    # any value above that gets wrapped around
    # the largest integer float32 can represent EXACT is 2^24, while for float64 it is 2^53
    x = np.asarray(x, dtype=np.float64)

    if x.size < t0+W+tau:
        raise IndexError("SizeError: End of signal reached")
    r_t0 = np.sum(x[t0:t0+W]*x[t0+tau:t0+W+tau])

    return r_t0

In [ ]:
# YIN algorithm

# Difference function
def diff(x, tau, W, t0=0):
    '''
    inputs:
        x[t]    - time-domain discrete audio signal (assuming 1D-np-array), 
        tau     - lag / candidate period, measured in samples
        t0      - starting sample of window W
        W       - window of autocorrelation function

    returns:
        difference function according to d_t(τ) = Σ_{j=0}^{W-1} ( x[t+j] − x[t+j+τ] )²
    '''

    x = np.asarray(x, dtype=np.float64)

    if x.size < t0+W+tau:
            raise IndexError("SizeError: End of signal reached")

    d_t = 0
    for j in range(W):
        d_t += (x[t0+j] - x[t0+j+tau])**2

        # for C++ implementation
        '''
        xa = x[t0+j]
        xb = x[t0+j+tau]
        d_t += (xa*xa - 2*xa*xb + xb*xb)
        '''

        # TODO -> fft-based implementation, based on YIN paper

    return d_t


# difference function for amplitude varying signal as by YIN paper
def diff_var_amp(x, tau, W, t0=0):
    '''
    d_t(tau) = r_t(0)*(1-r_t(tau)^2 /r_t(0) * r_(t+tau)(0)).
    '''

    x = np.asarray(x, dtype=np.float64)

    if x.size < t0+tau:
            raise IndexError("SizeError: End of signal reached")


    r_t_0 = acor(x, 0, W, t0)
    r_t_tau = acor(x, tau, W, t0)
    r_tplustau_0 = acor(x, 0, W, t0=t0+tau)

    if (r_t_0*r_tplustau_0)==0:
        return diff(x, tau, W, t0)

    d_t = r_t_0*(1-(r_t_tau**2) / (r_t_0*r_tplustau_0))
    
    return d_t

def diff_all(x,maxtau,W,t0=0, amp="static"):
    '''
    Chomputes and returns difference function for all tau up to maxtau.
    amp can be "variable" or "static"
    '''
    d = np.zeros(maxtau+1)

    if amp=="static":
        for i in range(maxtau+1):
            d[i] = diff(x,i,W,t0)

    elif amp=="variable":
        for i in range(maxtau+1):
            d[i] = diff_var_amp(x,i,W,t0)

    else:
        raise ValueError("amp argument must be either 'static' or 'variable'")

    return d

# Cumulative mean normalized difference (CMND)
def diff_cmnd(maxtau, diff_array):
    '''
    inputs:
        x[t]    - time-domain (sample-wise) discrete audio signal (assuming 1D-np-array), 
        tau     - lag / candidate period, measured in samples
        t0      - starting sample of window W
        W       - window of autocorrelation function

    returns:
        difference prime function according to  - d'(0) = 1
                                                - d'(τ) = d(τ) / [ (1/τ) · Σ_{j=1}^{τ} d(j) ]
                                                - d_prime includes d(a) for all values of a up to maxtau
    '''

    #d_t = np.ones(maxtau+1, dtype=np.float64)
    #d_t = diff_all(x, maxtau, W, t0)
    d_t = diff_array

    # d_t has the form [d'(0), d'(1), ..., d'(maxtau)], thus maxtau+1 elements

    d_prime = np.ones(maxtau+1, dtype=np.float64)
    #d_prime[1:] = d_t[np.arange(1, maxtau+1)]*np.arange(1, maxtau+1)/np.cumsum(d_t[1:])

    cs = np.cumsum(d_t[1:])
    # remove any 0/0 divisions and replace them by 1.0
    d_prime[1:] = np.divide(d_t[1:]*np.arange(1, maxtau+1), cs, out=np.ones(maxtau, np.float64), where=cs > 0)

    return d_prime


# absolute threshold
def abs_threshold(maxtau, diff_array, threshold=0.15):
    '''
    inputs:
        x[t]        - time-domain (sample-wise) discrete audio signal (assuming 1D-np-array), 
        tau         - lag / candidate period, measured in samples
        t0          - starting sample of window W
        W           - window of autocorrelation function
        threshold   - absolute threshold value for tau selection

    outputs:
        tau_min     - the local min after first tau to reach sub-threshold-value, or global argmin tau
        found       - was a local minimum found: True, or not: False
    '''

    #d_prime = diff_cmnd(x, maxtau, W, t0)
    d_prime = diff_array

    sub_th_idx = -1
    for i in range(maxtau+1):
        if d_prime[i] <= threshold:
            sub_th_idx = i
            break

    # if no tau reaches the threshold, choose global minimum
    if sub_th_idx == -1:
        tau_global_min = np.argmin(d_prime)
        return tau_global_min, False

    min_idx = -1
    for i in range(sub_th_idx, maxtau):
        if d_prime[i+1] >= d_prime[i]:
            min_idx = i
            break

    if min_idx == -1:
        return maxtau, True
    else:
        return min_idx, True


# parabolic interpolation
def par_interpol(maxtau, threshold, d_tau_plusone, d_prime):
    '''
    inputs:
        x[t]        - time-domain (sample-wise) discrete audio signal (assuming 1D-np-array), 
        tau         - lag / candidate period, measured in samples
        t0          - starting sample of window W
        W           - window of autocorrelation function
        threshold   - absolute threshold value for tau selection
    outputs:
        tau_star    - the parabolic approximation of the tau chosen in abs_threshold()
    '''

    d_chosen_tau_neighbors = d_tau_plusone

    chosen_tau, found_period = abs_threshold(maxtau, d_prime, threshold)

    # deal with 0/0 division
    if not found_period:
        return chosen_tau

    # τ* = τ + (d[τ−1] − d[τ+1]) / (2·(d[τ−1] − 2·d[τ] + d[τ+1]))
    tau_star = chosen_tau + (d_chosen_tau_neighbors[chosen_tau-1]-d_chosen_tau_neighbors[chosen_tau+1]) / (2*(d_chosen_tau_neighbors[chosen_tau-1]-2*d_chosen_tau_neighbors[chosen_tau]+d_chosen_tau_neighbors[chosen_tau+1]))

    return tau_star

In [ ]:
# voicing decision: is there a note in the frame or just noise/silence?
def yin_frame(x, maxtau, W, t0=0, threshold=0.15, rms_floor=1e-3):

    frame = np.asarray(x[t0:t0+W], dtype=np.float64)

    # too quiet: silence
    if np.sqrt(np.mean(frame**2)) < rms_floor:
        return 0.0

    d_tau_plusone = diff_all(x, maxtau+1, W, t0)
    d_prime = diff_cmnd(maxtau, d_tau_plusone[:-1])
    
    _, found = abs_threshold(maxtau, d_prime, threshold)

    # no clear period: noise, breath, attack
    if not found:
        return 0.0

    # period found, continue with YIN algorithm
    return fs / par_interpol(maxtau, threshold, d_tau_plusone, d_prime)

In [16]:
# test cases for the YIN functions: diff, diff_var_amp, diff_all, diff_cmnd, abs_threshold, par_interpol
# each test prints what it expected and what it got; a crash in one test does not stop the others
# sections A-F test one step each, section G runs the whole chain (yin_frame) on test tones and reports the pitch in Hz and cents,
# section H tests the voicing decision in yin_frame (np.nan = no pitch)
# the Python loops in diff() are slow, so this cell takes a minute or two

rng = np.random.default_rng(0)
results = []

def check(name, expected, got, tol=1e-9):
    '''compare numbers/arrays (relative tolerance) or anything else (==), print both, record PASS/FAIL'''
    try:
        e = np.asarray(expected, dtype=float)
        g = np.asarray(got, dtype=float)
        ok = e.shape == g.shape and np.allclose(g, e, rtol=tol, atol=tol)
    except (TypeError, ValueError):
        ok = expected == got
    results.append((name, ok))
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
    print(f"        expected: {expected}")
    print(f"        got:      {got}")

def run(name, test):
    '''run one test; an unexpected exception counts as FAIL and is printed'''
    try:
        test()
    except Exception as err:
        results.append((name, False))
        print(f"[FAIL] {name}")
        print(f"        raised {type(err).__name__}: {err}")


# ---------- independent reference versions (vectorized NumPy, written separately from the functions above) ----------

def ref_d(x, maxtau, W, t0=0):
    '''d(tau) = sum_j (x[t0+j] - x[t0+j+tau])^2 for tau = 0..maxtau'''
    x = np.asarray(x, dtype=np.float64); a = x[t0:t0+W]
    return np.array([np.sum((a - x[t0+k:t0+k+W])**2) for k in range(maxtau+1)])

def ref_d_var(x, maxtau, W, t0=0):
    '''amplitude-compensated difference: E(a) - <a,b>^2 / E(b), with a = window at t0, b = window at t0+tau'''
    x = np.asarray(x, dtype=np.float64); a = x[t0:t0+W]; out = np.zeros(maxtau+1)
    for k in range(1, maxtau+1):
        b = x[t0+k:t0+k+W]; out[k] = a @ a - (a @ b)**2 / (b @ b)
    return out

def ref_cmnd(d):
    '''d'(0) = 1, d'(tau) = d(tau) * tau / sum_{j=1}^{tau} d(j)'''
    out = np.ones_like(d); out[1:] = d[1:] * np.arange(1, len(d)) / np.cumsum(d[1:]); return out

def cents(f, f_true):
    return 1200 * np.log2(f / f_true)

fs = 44100                      # sample rate [Hz]
W = 1024                        # analysis window for the pitch tests (23 ms)
MAXTAU = 800                    # longest period searched: fs/800 = 55 Hz, just below the pedal Bb1 (58.27 Hz)
Bb1, Bb2, F3, Bb3, F4, F5 = 58.27, 116.54, 174.61, 233.08, 349.23, 698.46

AMPS = [0.3, 1.0, 0.9, 0.7, 0.5, 0.35, 0.25, 0.15]      # trombone-like: fundamental weaker than harmonics 2-4

def tone(f0, n, amps=(1.0,), env=1.0):
    '''sum of harmonics k*f0 with the given amplitudes, n samples long, times an envelope'''
    t = np.arange(n) / fs
    return env * sum(a * np.sin(2*np.pi*(k+1)*f0*t + 0.7*k) for k, a in enumerate(amps))

N = W + MAXTAU + 2             # enough samples for one frame at t0 = 0 (par_interpol reads up to t0+W+MAXTAU)

def pitch(x, t0=0, maxtau=MAXTAU):
    '''the whole chain with the voicing decision: pitch in Hz, or np.nan when yin_frame decides there is no note'''
    return yin_frame(x, maxtau, W, t0)


# ---------- A) diff ----------

# A1) matches the reference for several lags, window sizes and start samples
def t_diff_reference():
    x = rng.standard_normal(3000)
    for tau, Wt, t0 in [(0, 64, 0), (1, 64, 0), (37, 256, 10), (500, 1024, 777)]:
        check(f"A1 diff matches reference (tau={tau}, W={Wt}, t0={t0})", ref_d(x, tau, Wt, t0)[tau], diff(x, tau, Wt, t0))
run("A1 diff reference", t_diff_reference)

# A2) d(0) = 0: a window compared with itself has no difference
def t_diff_zero_lag():
    x = rng.standard_normal(2000)
    check("A2 diff at tau=0 is 0", 0.0, diff(x, 0, 512, 100))
run("A2 diff zero lag", t_diff_zero_lag)

# A3) sine with a whole-number period P: d(P) = 0 and d(P/2) = 4 * window energy (the two windows are sign-flipped)
def t_diff_integer_period():
    P = 100; x = np.sin(2*np.pi*np.arange(3000)/P)
    check("A3 sine with period 100: d(100) = 0", 0.0, round(diff(x, P, 1000), 9))
    check("A3 sine with period 100: d(50) = 4 * energy", 4*np.sum(x[:1000]**2), diff(x, P//2, 1000))
run("A3 diff integer period", t_diff_integer_period)

# A4) link to the autocorrelation: d(tau) = r_t(0) + r_{t+tau}(0) - 2 r_t(tau)   (eq. 7 in the YIN paper)
def t_diff_acor_identity():
    x = rng.standard_normal(2000); tau, Wt, t0 = 123, 400, 50
    check("A4 d(tau) = r_t(0) + r_(t+tau)(0) - 2 r_t(tau)",
          acor(x, 0, Wt, t0) + acor(x, 0, Wt, t0+tau) - 2*acor(x, tau, Wt, t0), diff(x, tau, Wt, t0))
run("A4 diff vs acor", t_diff_acor_identity)

# A5) input types: Python list, and loud int16 samples as read from a WAV file (no integer overflow)
def t_diff_dtypes():
    check("A5 Python list input", 2.0, diff([1., 2, 3, 4, 5], 1, 2))
    x16 = (np.sin(2*np.pi*np.arange(2000)/77.3) * 30000).astype(np.int16)
    check("A5 int16 input equals float input", ref_d(x16.astype(float), 40, 1024)[40], diff(x16, 40, 1024), tol=1e-9)
run("A5 diff dtypes", t_diff_dtypes)

# A6) end of signal: the last lag that fits works, one more raises IndexError
def t_diff_boundary():
    x = np.arange(10.0)
    check("A6 last fitting lag (t0+W+tau = len) works", ref_d(x, 7, 3)[7], diff(x, 7, 3))
    try:
        diff(x, 8, 3); got = "no error"
    except IndexError:
        got = "IndexError"
    check("A6 one lag past the end raises IndexError", "IndexError", got)
run("A6 diff boundary", t_diff_boundary)


# ---------- B) diff_var_amp ----------

# B1) matches the reference E(a) - <a,b>^2/E(b), including a window that does not start at 0
def t_var_reference():
    x = rng.standard_normal(4000)
    for tau, t0 in [(1, 0), (50, 0), (50, 2000), (378, 500)]:
        check(f"B1 diff_var_amp matches reference (tau={tau}, t0={t0})", ref_d_var(x, tau, 1024, t0)[tau], diff_var_amp(x, tau, 1024, t0))
run("B1 diff_var_amp reference", t_var_reference)

# B2) tau = 0 gives 0
def t_var_zero_lag():
    x = rng.standard_normal(2000)
    check("B2 diff_var_amp at tau=0 is 0", 0.0, round(diff_var_amp(x, 0, 512, 7), 9))
run("B2 diff_var_amp zero lag", t_var_zero_lag)

# B3) the point of this version: a sine whose amplitude grows by a factor g every period.
#     plain d(P) is large (the next period is louder), the amplitude-compensated one is ~0
def t_var_growing_sine():
    P = 100; n = np.arange(3000); x = 1.01**(n/P) * np.sin(2*np.pi*n/P)     # +1 % per period
    E = np.sum(x[:1000]**2)
    check("B3 growing sine: diff_var_amp(P) / energy ~ 0", 0.0, round(diff_var_amp(x, P, 1000) / E, 9))
    check("B3 growing sine: plain diff(P) / energy is clearly > 0  (showing True/False, value below)", True, diff(x, P, 1000) / E > 1e-5)
    print(f"        plain diff(P)/energy = {diff(x, P, 1000)/E:.2e}")
run("B3 diff_var_amp growing sine", t_var_growing_sine)

# B4) never negative (Cauchy-Schwarz: <a,b>^2 <= E(a) E(b)), checked over 200 random lags
def t_var_nonnegative():
    x = rng.standard_normal(3000)
    worst = min(diff_var_amp(x, tau, 512, 0) for tau in rng.integers(1, 2000, 200))
    check("B4 diff_var_amp >= 0 for 200 random lags (showing True/False, smallest value below)", True, bool(worst >= -1e-9))
    print(f"        smallest value = {worst:.3e}")
run("B4 diff_var_amp non-negative", t_var_nonnegative)

# B5) silence returns 0, not NaN
def t_var_silence():
    check("B5 diff_var_amp on silence is 0", 0.0, diff_var_amp(np.zeros(3000), 50, 1024))
run("B5 diff_var_amp silence", t_var_silence)


# ---------- C) diff_all ----------

# C1) static mode equals diff() for every lag, variable mode equals diff_var_amp() for every lag, length maxtau+1
def t_all_modes():
    x = rng.standard_normal(1000); m, Wt, t0 = 40, 256, 30
    d_s = diff_all(x, m, Wt, t0)
    d_v = diff_all(x, m, Wt, t0, amp="variable")
    check("C1 length is maxtau+1", m+1, len(d_s))
    check("C1 default ('static') equals diff() for tau = 0..40", [diff(x, k, Wt, t0) for k in range(m+1)], d_s)
    check("C1 'variable' equals diff_var_amp() for tau = 0..40", [diff_var_amp(x, k, Wt, t0) for k in range(m+1)], d_v)
run("C1 diff_all modes", t_all_modes)

# C2) an unknown mode raises ValueError
def t_all_bad_mode():
    try:
        diff_all(np.zeros(100), 5, 10, amp="loud"); got = "no error"
    except ValueError:
        got = "ValueError"
    check("C2 amp='loud' raises ValueError", "ValueError", got)
run("C2 diff_all bad mode", t_all_bad_mode)


# ---------- D) diff_cmnd ----------

# D1) matches the reference formula d'(tau) = d(tau) * tau / sum_{j=1}^{tau} d(j), and d'(0) = 1
def t_cmnd_reference():
    x = tone(Bb2, N, AMPS)
    got = diff_cmnd(x, 400, W)
    check("D1 d'(0) = 1", 1.0, got[0])
    check("D1 d' matches reference for tau = 0..400 (showing max |difference|)", 0.0, np.max(np.abs(got - ref_cmnd(ref_d(x, 400, W)))))
run("D1 cmnd reference", t_cmnd_reference)

# D2) white noise has no period: d' stays near 1 once tau is past a few samples
def t_cmnd_noise():
    x = rng.standard_normal(3000)
    m = np.mean(diff_cmnd(x, 400, W)[50:])
    check("D2 white noise: mean d' over tau = 50..400 within 0.8..1.2 (showing True/False, mean below)", True, bool(0.8 < m < 1.2))
    print(f"        mean d' = {m:.3f}")
run("D2 cmnd noise", t_cmnd_noise)

# D3) sine with a whole-number period P = 100: d'(P) = 0 and d' at the half period is large
def t_cmnd_integer_period():
    x = np.sin(2*np.pi*np.arange(3000)/100)
    dp = diff_cmnd(x, 150, 1000)
    check("D3 period 100: d'(100) = 0", 0.0, round(dp[100], 9))
    check("D3 period 100: d'(50) > 1  (showing True/False, value below)", True, bool(dp[50] > 1))
    print(f"        d'(50) = {dp[50]:.3f}")
run("D3 cmnd integer period", t_cmnd_integer_period)

# D4) silence: d' has no NaN (every d(tau) is 0, so the plain formula gives 0/0)
def t_cmnd_silence():
    dp = diff_cmnd(np.zeros(2000), 100, 512)
    check("D4 silence: no NaN in d'  (showing number of NaNs)", 0, int(np.sum(np.isnan(dp))))
run("D4 cmnd silence", t_cmnd_silence)


# ---------- E) abs_threshold ----------

# E1) trombone-like Bb2: returns the bottom of the first dip below 0.15, which is not the first sample below 0.15
def t_thr_bottom_of_dip():
    x = tone(Bb2, N, AMPS)
    dp = ref_cmnd(ref_d(x, 400, W))
    first_cross = int(np.argmax(dp <= 0.15))
    k = first_cross
    while dp[k+1] < dp[k]:
        k += 1
    got = abs_threshold(x, 400, W)[0]
    check(f"E1 Bb2: bottom of first dip (first crossing is at {first_cross})", k, got)
run("E1 threshold bottom of dip", t_thr_bottom_of_dip)

# E2) Bb2 plus noise at 10 dB SNR, 10 frames: never jumps to the dip at twice the period (an octave too low)
#     at 10 dB the dip at the period only reaches d' ~ 0.09-0.11, so with a threshold of 0.1 the deeper dip at 2x the period won sometimes
#     the noise also moves the integer tau by a few samples, so the check allows 5 % of the period (an octave error is 100 %)
def t_thr_no_octave():
    period = fs / Bb2; worst = 0
    for i in range(10):
        x = tone(Bb2, N + 5000, AMPS); x = x + np.sqrt(np.mean(x**2) / 10) * rng.standard_normal(x.size)
        tau, found = abs_threshold(x, MAXTAU, W, t0=500*i)
        worst = max(worst, abs(tau - period)) if found else np.inf
    check("E2 Bb2 + 10dB noise, 10 frames: dip found and tau within 5 % of the period (showing True/False, worst distance below)", True, bool(worst <= 0.05 * period))
    print(f"        worst distance = {worst:.2f} samples (period = {period:.2f})")
run("E2 threshold octave", t_thr_no_octave)

# E3) nothing gets below the threshold (white noise): falls back to the global minimum of d'
def t_thr_fallback():
    x = rng.standard_normal(3000)
    dp = diff_cmnd(x, 400, W)
    check("E3 white noise: returns argmin of d'", int(np.argmin(dp)), abs_threshold(x, 400, W)[0])
run("E3 threshold fallback", t_thr_fallback)

# E4) d' is still falling at maxtau (the period is just past the search range): returns maxtau
def t_thr_edge():
    x = np.sin(2*np.pi*np.arange(3000)/100)
    check("E4 period 100, maxtau 99: returns maxtau", 99, abs_threshold(x, 99, 1000)[0])
run("E4 threshold at maxtau", t_thr_edge)


# ---------- F) par_interpol ----------

# F1) matches the parabola formula applied to d(tau-1), d(tau), d(tau+1) around the tau from abs_threshold
def t_par_formula():
    x = tone(Bb2, N, AMPS)
    tau = abs_threshold(x, 400, W)[0]; d = ref_d(x, 401, W)
    exp = tau + (d[tau-1] - d[tau+1]) / (2*(d[tau-1] - 2*d[tau] + d[tau+1]))
    check(f"F1 Bb2: parabola through d around tau={tau}", exp, par_interpol(x, 400, W))
run("F1 par_interpol formula", t_par_formula)

# F2) the shift from the integer tau is at most half a sample
def t_par_half_sample():
    worst = 0
    for f in (Bb1, Bb2, F3, Bb3, F4, F5):
        x = tone(f, N)
        worst = max(worst, abs(par_interpol(x, MAXTAU, W) - abs_threshold(x, MAXTAU, W)[0]))
    check("F2 |tau* - tau| <= 0.5 on 6 sines (showing True/False, largest shift below)", True, bool(worst <= 0.5))
    print(f"        largest shift = {worst:.3f} samples")
run("F2 par_interpol half sample", t_par_half_sample)

# F3) chosen tau at the edge of the search range: returns a number instead of reading past the end of d
def t_par_edge():
    x = np.sin(2*np.pi*np.arange(3000)/100)
    got = par_interpol(x, 99, 1000)
    check("F3 period 100, maxtau 99: returns a finite number  (showing True/False, value below)", True, bool(np.isfinite(got)))
    print(f"        tau* = {got}")
run("F3 par_interpol at maxtau", t_par_edge)


# ---------- G) whole chain: pitch in Hz from yin_frame ----------

# G1) pure sines across the trombone range (pedal Bb1 to F5): error below 0.1 cent
def t_pitch_sines():
    for f in (Bb1, Bb2, F3, Bb3, F4, F5):
        got = pitch(tone(f, N))
        check(f"G1 sine {f} Hz: error < 0.1 cent  (showing True/False, pitch below)", True, bool(abs(cents(got, f)) < 0.1))
        print(f"        pitch = {got:.4f} Hz, error = {cents(got, f):+.4f} cents")
run("G1 sines", t_pitch_sines)

# G2) trombone-like tones (fundamental weaker than harmonics 2-4): error below 0.1 cent, no octave errors
def t_pitch_harmonic():
    for f in (Bb1, Bb2, F3, Bb3, F4):
        got = pitch(tone(f, N, AMPS))
        check(f"G2 trombone-like {f} Hz: error < 0.1 cent  (showing True/False, pitch below)", True, bool(abs(cents(got, f)) < 0.1))
        print(f"        pitch = {got:.4f} Hz, error = {cents(got, f):+.4f} cents")
run("G2 harmonic", t_pitch_harmonic)

# G3) Bb2 with white noise, 5 frames each. Every frame must stay voiced (it contains a note).
#     20 dB: every frame within 5 cents (a typical tuner tolerance).
#     10 dB: the noise moves single frames by up to ~18 cents (measured over 200 frames), so the limit is 20 cents per frame,
#            and the median of the 5 frames (what a smoothed tuner display would show) must be within 6 cents
def t_pitch_noise():
    for snr in (20, 10):
        x = tone(Bb2, N + 3000, AMPS); x = x + np.sqrt(np.mean(x**2) / 10**(snr/10)) * rng.standard_normal(x.size)
        f = np.array([pitch(x, t0=600*i) for i in range(5)])
        errs = cents(f, Bb2)
        check(f"G3 Bb2 + noise at {snr} dB SNR: all 5 frames voiced  (showing number of voiced frames)", 5, int(np.sum(~np.isnan(f))))
        limit = 5 if snr == 20 else 20
        check(f"G3 Bb2 + noise at {snr} dB SNR, 5 frames: every frame within {limit} cents  (showing True/False, errors below)", True, bool(np.nanmax(np.abs(errs)) < limit))
        print(f"        errors = {np.round(errs, 2)} cents")
        if snr == 10:
            med = cents(np.median(f), Bb2)
            check("G3 Bb2 + noise at 10 dB SNR: median of the 5 frames within 6 cents  (showing True/False, median error below)", True, bool(abs(med) < 6))
            print(f"        median error = {med:+.2f} cents")
run("G3 noise", t_pitch_noise)

# G4) crescendo: Bb2 getting 4x louder (+12 dB) within 100 ms, 5 frames along the swell: error below 1 cent
def t_pitch_crescendo():
    n = N + 5000; env = np.clip(0.25 + 7.5*np.arange(n)/fs, 0, 1)
    x = tone(Bb2, n, AMPS, env)
    errs = [cents(pitch(x, t0=1000*i), Bb2) for i in range(5)]
    check("G4 Bb2 crescendo +12 dB in 100 ms: error < 1 cent  (showing True/False, errors below)", True, bool(max(abs(e) for e in errs) < 1))
    print(f"        errors = {np.round(errs, 3)} cents")
run("G4 crescendo", t_pitch_crescendo)

# G5) int16 input (16-bit WAV samples) gives the same pitch as the float signal
def t_pitch_int16():
    xf = tone(Bb2, N, AMPS); x16 = np.round(xf / np.max(np.abs(xf)) * 30000).astype(np.int16)
    check("G5 int16 Bb2 gives the same pitch as float (within 0.01 cent)", pitch(xf), pitch(x16), tol=5e-6)
run("G5 int16", t_pitch_int16)

# ---------- H) voicing decision in yin_frame: np.nan means "no note in this frame" ----------

# H1) silence gives no pitch
def t_voice_silence():
    got = yin_frame(np.zeros(N), MAXTAU, W)
    check("H1 silence: yin_frame returns nan  (showing True/False, value below)", True, bool(np.isnan(got)))
    print(f"        yin_frame = {got}")
run("H1 silence", t_voice_silence)

# H2) a note far below the level floor (rms ~ 1e-4 < rms_floor = 1e-3, about -80 dBFS) gives no pitch
def t_voice_quiet():
    x = 1e-4 * tone(Bb2, N, AMPS) / np.sqrt(np.mean(tone(Bb2, N, AMPS)**2))
    got = yin_frame(x, MAXTAU, W)
    check("H2 Bb2 at rms 1e-4: yin_frame returns nan  (showing True/False, value below)", True, bool(np.isnan(got)))
    print(f"        yin_frame = {got}")
run("H2 quiet note", t_voice_quiet)

# H3) white noise (loud enough to pass the level floor, but no period), 5 frames: no pitch in any of them
def t_voice_noise():
    x = 0.1 * rng.standard_normal(N + 3000)
    got = np.array([yin_frame(x, MAXTAU, W, t0=600*i) for i in range(5)])
    check("H3 white noise, 5 frames: all nan  (showing number of frames with a pitch)", 0, int(np.sum(~np.isnan(got))))
run("H3 white noise", t_voice_noise)

# H4) the same note just above the level floor (rms 1e-2) still gets a pitch, and the right one
def t_voice_soft_note():
    x = 1e-2 * tone(Bb2, N, AMPS) / np.sqrt(np.mean(tone(Bb2, N, AMPS)**2))
    got = yin_frame(x, MAXTAU, W)
    check("H4 Bb2 at rms 1e-2: pitch within 0.1 cent  (showing True/False, pitch below)", True, bool(abs(cents(got, Bb2)) < 0.1))
    print(f"        yin_frame = {got}")
run("H4 soft note", t_voice_soft_note)

# H5) note onset: silence, then Bb2 starts (5 ms attack). Frames that end before the note are nan,
#     frames that lie fully inside the note have the right pitch. Frames in between are only printed.
def t_voice_onset():
    on = 3000; n = on + 4*W + MAXTAU + 2
    env = np.clip((np.arange(n) - on) / (0.005*fs), 0, 1)
    x = tone(Bb2, n, AMPS, env)
    before = yin_frame(x, MAXTAU, W, t0=on - W - MAXTAU - 200)    # the whole frame, including the lagged part, is silent
    inside = yin_frame(x, MAXTAU, W, t0=on + 400)                  # past the 5 ms attack
    check("H5 onset: frame before the note is nan  (showing True/False, value below)", True, bool(np.isnan(before)))
    print(f"        yin_frame = {before}")
    check("H5 onset: frame inside the note within 1 cent  (showing True/False, value below)", True, bool(abs(cents(inside, Bb2)) < 1))
    print(f"        yin_frame = {inside}")
    for t0 in range(on - W - 400, on + 1, 200):
        print(f"        straddling frame t0 = onset{t0-on:+d}: yin_frame = {yin_frame(x, MAXTAU, W, t0=t0)}")
run("H5 onset", t_voice_onset)


# summary
n_pass = sum(ok for _, ok in results)
print(f"\n{n_pass} / {len(results)} checks passed")
for name, ok in results:
    if not ok:
        print(f"  FAIL: {name}")


[PASS] A1 diff matches reference (tau=0, W=64, t0=0)
        expected: 0.0
        got:      0.0
[PASS] A1 diff matches reference (tau=1, W=64, t0=0)
        expected: 86.33269934962536
        got:      86.33269934962534
[PASS] A1 diff matches reference (tau=37, W=256, t0=10)
        expected: 580.6959417474964
        got:      580.6959417474965
[PASS] A1 diff matches reference (tau=500, W=1024, t0=777)
        expected: 2081.9510569777485
        got:      2081.9510569777485
[PASS] A2 diff at tau=0 is 0
        expected: 0.0
        got:      0.0
[PASS] A3 sine with period 100: d(100) = 0
        expected: 0.0
        got:      0.0
[PASS] A3 sine with period 100: d(50) = 4 * energy
        expected: 2000.0
        got:      2000.0
[PASS] A4 d(tau) = r_t(0) + r_(t+tau)(0) - 2 r_t(tau)
        expected: 767.913716176854
        got:      767.9137161768537
[PASS] A5 Python list input
        expected: 2.0
        got:      2.0
[PASS] A5 int16 input equals float input
        expected: 